In [1]:
!pip install flash-attn sympy math_verify pylatexenc


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
from vllm import LLM, SamplingParams

# Create an LLM.
llm = LLM(model='sft_model_ei_single')

INFO 08-31 22:22:08 [__init__.py:244] Automatically detected platform rocm.
INFO 08-31 22:22:25 [config.py:853] This model supports multiple tasks: {'embed', 'score', 'classify', 'reward', 'generate'}. Defaulting to 'generate'.
INFO 08-31 22:22:25 [config.py:1467] Using max model len 4096
INFO 08-31 22:22:33 [config.py:2267] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 08-31 22:22:33 [config.py:4566] full_cuda_graph is not supported with cascade attention. Disabling cascade attention.
WARNING 08-31 22:22:34 [utils.py:2613] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reason: CUDA is initialized
INFO 08-31 22:22:37 [__init__.py:244] Automatically detected platform rocm.
INFO 08-31 22:22:46 [core.py:459] Waiting for init message from front-end.
INFO 08-31 22:22:46 [core.py:69] Initializing a V1 LLM en

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.10s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.11s/it]



INFO 08-31 22:22:47 [default_loader.py:272] Loading weights took 1.21 seconds
INFO 08-31 22:22:48 [gpu_model_runner.py:1782] Model loading took 3.0352 GiB and 1.353108 seconds
INFO 08-31 22:22:52 [backends.py:509] Using cache directory: /root/.cache/vllm/torch_compile_cache/e8bb18619c/rank_0_0/backbone for vLLM's torch.compile
INFO 08-31 22:22:52 [backends.py:520] Dynamo bytecode transform time: 4.18 s
INFO 08-31 22:22:54 [backends.py:155] Directly load the compiled graph(s) for shape None from the cache, took 0.477 s
INFO 08-31 22:22:55 [monitor.py:34] torch.compile takes 4.18 s in total
INFO 08-31 22:23:07 [gpu_worker.py:232] Available KV cache memory: 163.12 GiB
INFO 08-31 22:23:08 [kv_cache_utils.py:716] GPU KV cache size: 6,108,624 tokens
INFO 08-31 22:23:08 [kv_cache_utils.py:720] Maximum concurrency for 4,096 tokens per request: 1491.36x
INFO 08-31 22:23:08 [rocm.py:224] Using Triton Attention backend on V1 engine.


Capturing CUDA graphs: 100%|██████████| 67/67 [00:08<00:00,  8.22it/s]


INFO 08-31 22:23:16 [gpu_model_runner.py:2306] Graph capturing finished in 8 secs, took 0.27 GiB
INFO 08-31 22:23:16 [core.py:172] init engine (profile, create kv cache, warmup model) took 28.10 seconds


In [3]:
question = "Simplify $(3-i)(6+2i)$." 

# Sample prompts.
prompts = [
    f"""A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>""",
]
sampling_min_tokens = 4
G = 2 # number of response generated

# Create a sampling params object, stopping generation on newline.
sampling_params = SamplingParams(
    temperature=1.0, 
    top_p=1.0, 
    max_tokens=1024, 
    stop=["\n"], 
    min_tokens=sampling_min_tokens,
    n=G
)

sampling_params.stop = ["</answer>"]
sampling_params.include_stop_str_in_output = True

# Generate texts from the prompts. The output is a list of RequestOutput objects
# that contain the prompt, generated text, and other information.
outputs = llm.generate(prompts, sampling_params)

# Print the outputs.
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}")
    print(f"Generated text: {generated_text!r}")

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0% 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Prompt: 'A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Simplify $(3-i)(6+2i)$.\nAssistant: <think>'
Generated text: "Alright, let's simplify the expression (3 - i)(6 + 2i). Hmm, I need to remember how to multiply complex numbers. I remember that when you multiply two complex numbers, you use the distributive property, right? Let me recall. If I have (a + bi)(c + di), the result should be ac + adi + bci + bdi². But since i² is -1, that term simplifies to -bd. So the formula should be ac + adi + bci - bd. Let me write that down to make sure: (a + bi)(c + di) = ac + adi + bci - bd. Now, applying this to my problem. Here,

In [6]:
from drgrpo_grader import r1_zero_reward_fn


In [7]:
from typing import Callable

def expert_iteration_eval(
  vllm_model: LLM,
  reward_fn: Callable[[str, str], dict[str, float]],
  prompts: list[str],
  answers: list[str],
  eval_sampling_params: SamplingParams
) :
  """
  Evaluate a language model on a list of prompts,
  compute evaluation metrics, and serialize results to disk.
  """
  outputs = vllm_model.generate(prompts, eval_sampling_params)
  training_data = []
      
  for index, output in enumerate(outputs):
    prompt = output.prompt
    generated_text_0 = output.outputs[0].text
    generated_text_1 = output.outputs[1].text
    for i in range(G):
        generated_text_i = output.outputs[i].text
        if (reward_fn(generated_text_i, answers[index])['reward'] == 1.0):
            training_data.append([prompt, generated_text_0])
            break
            
  return training_data

In [ ]:
import pandas as pd
from tqdm import tqdm
import random 

df = pd.read_parquet("math_12k.parquet")

total_format_reward = 0.
total_answer_reward = 0.
total_reward = 0.
prompts = []
answers = []
dataset = []

for index, row in tqdm(df.iterrows()):
    question = row['problem']
    answer = row['solution']
    prompt = f"""A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>"""
    prompts.append(prompt)
    answers.append(answer)

    if (index + 1) % 240 == 0:
        dataset += expert_iteration_eval(llm, r1_zero_reward_fn, prompts, answers, sampling_params)

        prompts = []
        answers = []

0it [00:00, ?it/s]

Adding requests:   0%|          | 0/240 [00:00<?, ?it/s]

Processed prompts:   0% 0/480 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

240it [00:13, 17.59it/s]

Adding requests:   0%|          | 0/240 [00:00<?, ?it/s]

...

Adding requests:   0%|          | 0/240 [00:00<?, ?it/s]

Processed prompts:   0% 0/480 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

2640it [02:52, 14.58it/s]

Adding requests:   0%|          | 0/240 [00:00<?, ?it/s]

Processed prompts:   0% 0/480 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [ ]:
len(dataset)

In [17]:
df = pd.DataFrame(dataset, columns=['prompt', 'response'])
df.to_parquet('expert_iteration_training_single_1.parquet', index=False)

In [18]:
df

,prompt,response
0,A conversation between User and Assistant. The...,"Okay, let's start by understanding the proble..."
1,A conversation between User and Assistant. The...,"\n\nHmm, let me tackle this problem step by s..."
2,A conversation between User and Assistant. The...,"Okay, let's break this down step by step. Fir..."
3,A conversation between User and Assistant. The...,"Okay, so the odds of pulling a prize out of th..."
4,A conversation between User and Assistant. The...,"Okay, so each die has six sides, numbered 1 t..."
...,...,...
4408,A conversation between User and Assistant. The...,"Alright, so I need to find the value of y such..."
4409,A conversation between User and Assistant. The...,"Alright, let's find the length of the line se..."
4410,A conversation between User and Assistant. The...,"Okay, let's start by simplifying the denomina..."
4411,A conversation between User and Assistant. The...,"Okay, let me try to figure out when \(P(G(a))\..."
